In [1]:
# ============================================
# INSTALL
# ============================================

!pip install monai nibabel --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 28.3 MB/s eta 0:00:0000:0100:01


In [13]:
import os
print(os.listdir('/kaggle/input/datasets/gautamganesh03/deep-bet-output/DeepBET_Output'))

['OAS2_0159', 'OAS2_0106', 'OAS2_0165', 'OAS2_0171', 'OAS2_0172', 'OAS2_0017', 'OAS2_0087', 'OAS2_0035', 'OAS2_0071', 'OAS2_0175', 'OAS2_0086', 'OAS2_0069', 'OAS2_0181', 'OAS2_0185', 'OAS2_0043', 'OAS2_0054', 'OAS2_0134', 'OAS2_0135', 'OAS2_0158', 'OAS2_0057', 'OAS2_0127', 'OAS2_0028', 'OAS2_0154', 'OAS2_0116', 'OAS2_0162', 'OAS2_0014', 'OAS2_0031', 'OAS2_0041', 'OAS2_0077', 'OAS2_0002', 'OAS2_0089', 'OAS2_0032', 'OAS2_0147', 'OAS2_0176', 'OAS2_0164', 'OAS2_0118', 'OAS2_0144', 'OAS2_0040', 'OAS2_0122', 'OAS2_0113', 'OAS2_0169', 'OAS2_0064', 'OAS2_0145', 'OAS2_0030', 'OAS2_0124', 'OAS2_0090', 'OAS2_0096', 'OAS2_0068', 'OAS2_0012', 'OAS2_0081', 'OAS2_0102', 'OAS2_0070', 'OAS2_0048', 'OAS2_0095', 'OAS2_0152', 'OAS2_0046', 'OAS2_0149', 'OAS2_0157', 'OAS2_0091', 'OAS2_0034', 'OAS2_0061', 'OAS2_0027', 'OAS2_0010', 'OAS2_0141', 'OAS2_0062', 'OAS2_0126', 'OAS2_0055', 'OAS2_0111', 'OAS2_0146', 'OAS2_0112', 'OAS2_0161', 'OAS2_0140', 'OAS2_0053', 'OAS2_0039', 'OAS2_0026', 'OAS2_0016', 'OAS2_0049'

In [14]:
# ============================================
# FAST HYBRID MRI + XGBOOST PIPELINE
# FINAL KAGGLE VERSION
# ============================================

# ============================================
# IMPORTS
# ============================================

import gc
import random
import warnings
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier

from monai.networks.nets import resnet
from monai.transforms import Compose, RandAffine

warnings.filterwarnings('ignore')


# ============================================
# REPRODUCIBILITY
# ============================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# ============================================
# CONFIG
# ============================================

MRI_PATH = Path(
    '/kaggle/input/datasets/gautamganesh03/deep-bet-output/DeepBET_Output'
)

DEMOGRAPHICS_PATH = (
    '/kaggle/input/datasets/gautamganesh03/clinical-data/'
    'cleaned_demographics.csv'
)

SAVE_DIR = Path('/kaggle/working/OASIS_RESULTS')

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)



TARGET_SHAPE = (64, 64, 64)

BATCH_SIZE = 4

NUM_WORKERS = 2

EPOCHS = 30

EARLY_STOPPING_PATIENCE = 5

MIN_DELTA = 0.01

LEARNING_RATE = 2e-4

N_SPLITS = 4

EMBEDDING_DIM = 64


# ============================================
# DEVICE
# ============================================

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu'
)

print('Device:', device)

if torch.cuda.is_available():

    print(
        torch.cuda.get_device_name(0)
    )

    torch.backends.cudnn.benchmark = True


# ============================================
# CLASS NAMES
# ============================================

severity_names = {

    0: 'Non Demented',

    1: 'Very Mild',

    2: 'Demented'
}


# ============================================
# LOAD CLEANED DEMOGRAPHICS
# ============================================

df_demo = pd.read_csv(
    DEMOGRAPHICS_PATH
)

print(
    '\nLoaded rows:',
    len(df_demo)
)


# ============================================
# CLINICAL FEATURES
# ============================================

clinical_features = [

    'Age',

    'MMSE',

    'SES',

    'EDUC',

    'nWBV',

    'eTIV',

    'ASF',

    'M/F'
]


# ============================================
# CLASS DISTRIBUTION
# ============================================

print('\nSeverity Distribution:\n')

print(
    df_demo['Severity']
    .value_counts()
)


# ============================================
# MRI AUGMENTATION
# ============================================

train_transforms = Compose([

    RandAffine(

        prob=0.25,

        rotate_range=(
            0.05,
            0.05,
            0.05
        ),

        scale_range=(
            0.05,
            0.05,
            0.05
        ),

        padding_mode='border'
    )

])


# ============================================
# FIND MRI SCANS
# ============================================

def find_all_scans(root):

    scans = []

    unmatched_patients = []

    for patient_folder in sorted(
        root.iterdir()
    ):

        if not patient_folder.is_dir():
            continue

        patient_id = patient_folder.name

        subject_rows = df_demo[

            df_demo['Clean_ID']
            == patient_id
        ]

        if len(subject_rows) == 0:

            unmatched_patients.append(
                patient_id
            )

            continue

        subject_row = subject_rows.iloc[0]

        severity = int(
            subject_row['Severity']
        )

        clinical_vector = (

            subject_row[
                clinical_features
            ]

            .values

            .astype(np.float32)
        )

        for label_folder in sorted(
            patient_folder.iterdir()
        ):

            if not label_folder.is_dir():
                continue

            brain_folder = (
                label_folder / 'brain'
            )

            if not brain_folder.exists():
                continue

            scan_files = (

                list(
                    brain_folder.glob('*.nii')
                )

                +

                list(
                    brain_folder.glob(
                        '*.nii.gz'
                    )
                )
            )

            for scan_file in scan_files:

                scans.append({

                    'patient_id':
                    patient_id,

                    'filepath':
                    scan_file,

                    'severity':
                    severity,

                    'clinical':
                    clinical_vector
                })

    print(
        '\nUnmatched patients:',
        len(unmatched_patients)
    )

    return scans


all_scans = find_all_scans(
    MRI_PATH
)

print(
    '\nTotal scans:',
    len(all_scans)
)


# ============================================
# PREPROCESSING
# ============================================

def preprocess_volume(filepath):

    img = nib.load(
        str(filepath)
    )

    volume = img.get_fdata()

    volume = volume.astype(
        np.float32
    )

    while volume.ndim > 3:

        volume = volume.squeeze(-1)

    mean = volume.mean()

    std = volume.std()

    if std > 0:

        volume = (
            volume - mean
        ) / std

    volume = torch.tensor(volume)

    volume = (

        volume

        .unsqueeze(0)

        .unsqueeze(0)
    )

    volume = F.interpolate(

        volume,

        size=TARGET_SHAPE,

        mode='trilinear',

        align_corners=False
    )

    volume = volume.squeeze(0)

    return volume


# ============================================
# DATASET
# ============================================

class OASISDataset(Dataset):

    def __init__(
        self,
        scan_list,
        transforms=None
    ):

        self.scan_list = scan_list

        self.transforms = transforms

    def __len__(self):

        return len(
            self.scan_list
        )

    def __getitem__(self, idx):

        sample = self.scan_list[idx]

        volume = preprocess_volume(
            sample['filepath']
        )

        if self.transforms:

            volume = (

                self.transforms(volume)

                .float()
            )

        label = torch.tensor(

            sample['severity'],

            dtype=torch.long
        )

        clinical = torch.tensor(

            sample['clinical'],

            dtype=torch.float32
        )

        return (

            volume,

            label,

            clinical,

            sample['patient_id']
        )


# ============================================
# MODEL
# ============================================

class HybridResNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = (
            resnet.resnet18(
                spatial_dims=3,
                n_input_channels=1,
                num_classes=3
            )
        )

        # Freeze everything
        for param in (
            self.backbone.parameters()
        ):
            param.requires_grad = False

        # Train ONLY layer4
        for param in (
            self.backbone
            .layer4
            .parameters()
        ):
            param.requires_grad = True

        feature_dim = (
            self.backbone
            .fc
            .in_features
        )

        self.backbone.fc = nn.Identity()

        self.embedding_head = nn.Sequential(

            nn.Linear(
                feature_dim,
                EMBEDDING_DIM
            ),

            nn.ReLU(),

            nn.Dropout(0.4)
        )

        self.classifier = nn.Linear(
            EMBEDDING_DIM,
            3
        )

    def forward(self, x):

        features = self.backbone(x)

        embeddings = (
            self.embedding_head(
                features
            )
        )

        logits = self.classifier(
            embeddings
        )

        return logits, embeddings


# ============================================
# PATIENT SPLITS
# ============================================

patient_to_label = {}

for scan in all_scans:

    patient_to_label[
        scan['patient_id']
    ] = scan['severity']

patients = np.array(
    list(patient_to_label.keys())
)

labels = np.array(
    list(patient_to_label.values())
)


# ============================================
# CROSS VALIDATION
# ============================================

sgkf = StratifiedGroupKFold(

    n_splits=N_SPLITS,

    shuffle=True,

    random_state=SEED
)

fold_accuracies = []


# ============================================
# TRAINING LOOP
# ============================================

for fold, (

    train_idx,

    val_idx

) in enumerate(

    sgkf.split(

        patients,

        labels,

        groups=patients
    )

):

    print('\n' + '=' * 60)

    print(f'FOLD {fold+1}')

    print('=' * 60)

    train_patients = (
        patients[train_idx]
    )

    val_patients = (
        patients[val_idx]
    )

    train_scans = [

        s for s in all_scans

        if s['patient_id']
        in train_patients
    ]

    val_scans = [

        s for s in all_scans

        if s['patient_id']
        in val_patients
    ]

    print(
        'Train scans:',
        len(train_scans)
    )

    print(
        'Validation scans:',
        len(val_scans)
    )


    # ========================================
    # DATASETS
    # ========================================

    train_dataset = OASISDataset(
        train_scans,
        transforms=train_transforms
    )

    val_dataset = OASISDataset(
        val_scans
    )


    # ========================================
    # DATALOADERS
    # ========================================

    train_loader = DataLoader(

        train_dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        num_workers=NUM_WORKERS,

        pin_memory=True
    )

    val_loader = DataLoader(

        val_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=NUM_WORKERS,

        pin_memory=True
    )


    # ========================================
    # MODEL
    # ========================================

    model = HybridResNet().to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(

        filter(
            lambda p:
            p.requires_grad,

            model.parameters()
        ),

        lr=LEARNING_RATE
    )

    scaler = torch.amp.GradScaler('cuda')

    best_val_loss = float('inf')

    best_epoch = 0

    patience_counter = 0

    best_model_path = (

        SAVE_DIR /

        f'best_model_fold_{fold+1}.pth'
    )


    # ========================================
    # TRAINING
    # ========================================

    for epoch in range(EPOCHS):

        # ====================================
        # TRAIN
        # ====================================

        model.train()

        train_loss = 0

        for (

            volumes,

            labels_batch,

            _,

            _

        ) in train_loader:

            volumes = volumes.to(device)

            labels_batch = (
                labels_batch.to(device)
            )

            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):

                outputs, _ = model(
                    volumes
                )

                loss = criterion(
                    outputs,
                    labels_batch
                )

            scaler.scale(loss).backward()

            scaler.step(optimizer)

            scaler.update()

            train_loss += loss.item()


        # ====================================
        # VALIDATION
        # ====================================

        model.eval()

        val_loss = 0

        val_preds = []

        val_true = []

        with torch.no_grad():

            for (

                volumes,

                labels_batch,

                _,

                _

            ) in val_loader:

                volumes = volumes.to(device)

                labels_batch = (
                    labels_batch.to(device)
                )

                outputs, _ = model(
                    volumes
                )

                loss = criterion(
                    outputs,
                    labels_batch
                )

                val_loss += loss.item()

                preds = torch.argmax(
                    outputs,
                    dim=1
                )

                val_preds.extend(
                    preds.cpu().numpy()
                )

                val_true.extend(
                    labels_batch
                    .cpu()
                    .numpy()
                )

        val_loss /= len(val_loader)

        val_acc = accuracy_score(
            val_true,
            val_preds
        )

        print(

            f'Fold {fold+1} | '

            f'Epoch {epoch+1}/{EPOCHS} | '

            f'Train Loss: '
            f'{train_loss/len(train_loader):.4f} | '

            f'Val Loss: '
            f'{val_loss:.4f} | '

            f'Val Acc: '
            f'{val_acc:.4f}'
        )


        # ====================================
        # SAVE BEST MODEL
        # ====================================

        if val_loss < (
            best_val_loss - MIN_DELTA
        ):

            best_val_loss = val_loss

            best_epoch = epoch + 1

            patience_counter = 0

            print(

                f'\nValidation improved '

                f'significantly. '

                f'Saving best model '

                f'from Epoch '

                f'{best_epoch} '

                f'with Val Loss '

                f'{best_val_loss:.4f}'
            )

            torch.save(

                model.state_dict(),

                best_model_path
            )

        else:

            patience_counter += 1

            print(

                f'No significant improvement. '

                f'Patience: '

                f'{patience_counter}/'

                f'{EARLY_STOPPING_PATIENCE}'
            )


        # ====================================
        # EARLY STOPPING
        # ====================================

        if (
            patience_counter >=
            EARLY_STOPPING_PATIENCE
        ):

            print(
                '\nEarly stopping triggered.'
            )

            break


    print(

        f'\nBest model for Fold '

        f'{fold+1} came from '

        f'Epoch {best_epoch} '

        f'with Validation Loss '

        f'{best_val_loss:.4f}'
    )


    # ========================================
    # LOAD BEST MODEL
    # ========================================

    model.load_state_dict(

        torch.load(

            best_model_path,

            map_location=device,

            weights_only=True
        )
    )

    model.eval()


    # ========================================
    # EXTRACT EMBEDDINGS
    # ========================================

    def extract_embeddings(loader):

        embeddings_all = []

        labels_all = []

        clinical_all = []

        patient_ids_all = []

        with torch.no_grad():

            for (

                volumes,

                labels_batch,

                clinical_batch,

                patient_ids

            ) in loader:

                volumes = volumes.to(device)

                _, embeddings = model(
                    volumes
                )

                embeddings_all.append(
                    embeddings
                    .cpu()
                    .numpy()
                )

                labels_all.append(
                    labels_batch.numpy()
                )

                clinical_all.append(
                    clinical_batch.numpy()
                )

                patient_ids_all.extend(
                    patient_ids
                )

        embeddings_all = np.concatenate(
            embeddings_all
        )

        labels_all = np.concatenate(
            labels_all
        )

        clinical_all = np.concatenate(
            clinical_all
        )

        return (
            embeddings_all,
            labels_all,
            clinical_all,
            patient_ids_all
        )


    (
        train_embeddings,
        train_labels,
        train_clinical,
        train_patient_ids

    ) = extract_embeddings(
        train_loader
    )

    (
        val_embeddings,
        val_labels,
        val_clinical,
        val_patient_ids

    ) = extract_embeddings(
        val_loader
    )


    # ========================================
    # SAVE EMBEDDINGS
    # ========================================

    np.save(

        SAVE_DIR /

        f'fold_{fold+1}_train_embeddings.npy',

        train_embeddings
    )

    np.save(

        SAVE_DIR /

        f'fold_{fold+1}_val_embeddings.npy',

        val_embeddings
    )


    # ========================================
    # COMBINE FEATURES
    # ========================================

    X_train = np.concatenate(

        [
            train_embeddings,
            train_clinical
        ],

        axis=1
    )

    X_val = np.concatenate(

        [
            val_embeddings,
            val_clinical
        ],

        axis=1
    )


    # ========================================
    # SCALE FEATURES
    # ========================================

    scaler_tabular = (
        StandardScaler()
    )

    X_train = (

        scaler_tabular
        .fit_transform(X_train)
    )

    X_val = (

        scaler_tabular
        .transform(X_val)
    )


    # ========================================
    # XGBOOST
    # ========================================

    xgb_model = XGBClassifier(

        n_estimators=250,

        max_depth=4,

        learning_rate=0.05,

        subsample=0.8,

        colsample_bytree=0.8,

        objective='multi:softmax',

        num_class=3,

        eval_metric='mlogloss',

        random_state=SEED,

        device='cuda'
        if torch.cuda.is_available()
        else 'cpu'
    )

    xgb_model.fit(
        X_train,
        train_labels
    )

    preds = xgb_model.predict(
        X_val
    )

    fold_acc = accuracy_score(
        val_labels,
        preds
    )

    fold_accuracies.append(
        fold_acc
    )

    print('\nXGBoost Accuracy:')

    print(fold_acc)

    print('\nClassification Report:\n')

    print(

        classification_report(

            val_labels,

            preds,

            target_names=[
                'Non Demented',
                'Very Mild',
                'Demented'
            ]
        )
    )

    print('\nConfusion Matrix:\n')

    print(
        confusion_matrix(
            val_labels,
            preds
        )
    )


    # ========================================
    # MEMORY CLEANUP
    # ========================================

    del model

    gc.collect()

    torch.cuda.empty_cache()


# ============================================
# FINAL RESULTS
# ============================================

print('\n' + '=' * 60)

print('FINAL CROSS VALIDATION RESULTS')

print('=' * 60)

for i, acc in enumerate(
    fold_accuracies
):

    print(
        f'Fold {i+1}: {acc:.4f}'
    )

print(
    '\nMean Accuracy:',
    np.mean(fold_accuracies)
)

print(
    'Std Accuracy:',
    np.std(fold_accuracies)
)

Device: cuda
Tesla T4

Loaded rows: 373

Severity Distribution:

Severity
0    206
1    123
2     44
Name: count, dtype: int64

Unmatched patients: 0

Total scans: 1368

FOLD 1
Train scans: 1022
Validation scans: 346
Fold 1 | Epoch 1/30 | Train Loss: 0.8573 | Val Loss: 2.0835 | Val Acc: 0.5549

Validation improved significantly. Saving best model from Epoch 1 with Val Loss 2.0835
Fold 1 | Epoch 2/30 | Train Loss: 0.6409 | Val Loss: 1.9231 | Val Acc: 0.5665

Validation improved significantly. Saving best model from Epoch 2 with Val Loss 1.9231
Fold 1 | Epoch 3/30 | Train Loss: 0.4486 | Val Loss: 2.0234 | Val Acc: 0.1908
No significant improvement. Patience: 1/5
Fold 1 | Epoch 4/30 | Train Loss: 0.3448 | Val Loss: 1.6290 | Val Acc: 0.6069

Validation improved significantly. Saving best model from Epoch 4 with Val Loss 1.6290
Fold 1 | Epoch 5/30 | Train Loss: 0.2708 | Val Loss: 4.9022 | Val Acc: 0.5520
No significant improvement. Patience: 1/5
Fold 1 | Epoch 6/30 | Train Loss: 0.2369 | Va

In [ ]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler


train_embeddings = np.load('OASIS_RESULTS/fold_1_train_embeddings.npy')
train_clinical = np.load('OASIS_RESULTS/fold_1_train_clinical.npy')
train_labels = np.load('OASIS_RESULTS/fold_1_train_labels.npy')

val_embeddings = np.load('OASIS_RESULTS/fold_1_val_embeddings.npy')
val_clinical = np.load('OASIS_RESULTS/fold_1_val_clinical.npy')
val_labels = np.load('OASIS_RESULTS/fold_1_val_labels.npy')


X_train = np.concatenate([train_embeddings, train_clinical], axis=1)
X_val = np.concatenate([val_embeddings, val_clinical], axis=1)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)


xgb_model = XGBClassifier(n_estimators=500, max_depth=5, learning_rate=0.01) 
xgb_model.fit(X_train_scaled, train_labels)